In [6]:

%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
root_path = "/Users/brianlerner/Library/CloudStorage/OneDrive-DukeUniversity/Code/Ewandi/src"
# root_path = str(Path(__file__).resolve().parent.parent)
sys.path.append(root_path)


from sim import construct
from server.transfer import insert_profile_data
from server.remove import remove_profile
from server import connect_to_collection



The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Dropping collection: profiles
Operation completed.


Here's the plan- we need to get all the names of possible things to compare to.
I'll plan to extract people separately from the user prompt so i don't need to make combo ids for that, but I will want to extract the people that are present

# Scores

`
{'name': 'mood',
 'all_day': True,
 'category': 'scores',
 'id': 'mood',
 'start_datetime': datetime.datetime(2023, 9, 1, 0, 0),
 'end_datetime': datetime.datetime(2023, 9, 2, 0, 0),
 'values': [{'name': 'score', 'value': 7}]}
`
Right now, I'm only going to use id.

## Sensations

Not doing intensity, so only need to worry about the id.

In [21]:
import random

# Example dictionary with more than 10 members
example_dict = {
    "member1": "value1",
    "member2": "value2",
    "member3": "value3",
    "member4": "value4",
    "member5": "value5",
    "member6": "value6",
    "member7": "value7",
    "member8": "value8",
    "member9": "value9",
    "member10": "value10",
    "member11": "value11",
    "member12": "value12",
    "member13": "value13",
    "member14": "value14",
    "member15": "value15"
}



# Using the function to get a dictionary with 10 random members
selected_dict = select_random_members_as_dict(example_dict)
selected_dict





{'member15': 'value15',
 'member3': 'value3',
 'member4': 'value4',
 'member10': 'value10',
 'member5': 'value5',
 'member6': 'value6',
 'member14': 'value14',
 'member12': 'value12',
 'member11': 'value11',
 'member9': 'value9'}

In [13]:
remove_profile()
construct()
insert_profile_data()

Removed profile: llm_v1
Precedence conflict:weightlifting, basketball_practice
Precedence conflict:weightlifting, basketball_practice
Precedence conflict:date_night, movie_night
Precedence conflict:party, class_computer_programming
Precedence conflict:date_night, movie_night
Precedence conflict:weightlifting, basketball_practice
Precedence conflict:weightlifting, basketball_practice
Precedence conflict:date_night, class_computer_programming
Precedence conflict:weightlifting, basketball_practice
Precedence conflict:party, class_computer_programming
Precedence conflict:date_night, movie_night
Precedence conflict:weightlifting, basketball_practice
Precedence conflict:party, class_computer_programming
Precedence conflict:weightlifting, basketball_practice
Precedence conflict:weightlifting, basketball_practice
Precedence conflict:party, class_computer_programming
Precedence conflict:weightlifting, basketball_practice
Precedence conflict:weightlifting, basketball_practice
Precedence conflict

In [39]:
a = '1'
b = a
b+= '2'
a

'1'

In [ ]:
{'a'}

In [14]:


from collections import defaultdict
from make_embedding_test import get_all_events_by_cat



events = get_all_events_by_cat()
names = get_all_names(events)
for cat, cat_names in names.items():
    print(cat, len(cat_names))
    for nm in cat_names:
        print(nm, ", ", cat_names[nm]['description']) 
    print()
# combine scheduled  unscheduled sleep

scheduled 9
basketball game ,  Playing in game for the Duke basketball team.
basketball practice ,  Required practice for the Duke basketball team.
class: Computer Programming ,  A class on computer programming at Duke University.
class: Discrete Mathematics ,  A class on discrete mathematics at Duke University.
date night ,  Going on a date night.
homework due: Computer Programming ,  The due day for the current homework.
homework due: Discrete Mathematics ,  The due day for the current homework.
movie night ,  Movie night at the theater with friends.
weightlifting ,  Required weightlifting for the Duke basketball team.

people 3
Mara (girlfriend) ,  Long-term girlfriend.
Mom (mother) ,  None
Tyrese (friend) ,  Friend from the basketball team.

unscheduled 6
hiking ,  Going hiking at Eno River State Park.
playing videogames ,  Playing videogames.
reading ,  Reading a book or magazine.
studying ,  Studying for classes.
talking on the phone ,  Having a conversation on the phone.
walking

For occurrences, it is sensible to group different labels together. For quantitative values, such as sleep duration, it doesn't really make sense to take more than one value. But what if someone says, how did my stress and mood behave over a time period? Then we'll break the problem down into chunks and apply the algorithm. 

So, in terms of needing groups, I don't really care about generating these for the quant or rubric_values, but I do want to get variations on their names.

I have events in the following categories: ['true_events', 'sensations', 'scores']


I am building a keyword matching system, and need assistance in generating code. Below, I will describe my plan and provide my current code. I would like you generate code for the full plan, by building off the current code using information from the described plan.
Please make code well-documented and pythonic.

--- Plan ----


I have a method to extract keywords from a user's input. I would then like to check and see if this keyword is semantically related to names in my database. A keyword can be related to either a name or a group, while a group can relate to 1 or more names. However, since I only have names in the database, I need to generate the groups.

The names are stored in a dictionary, where the key is a category. I would like to generate an encompassing set of groups for the names in a category. This should be performed by prompting an LLM. When the groups are generated for each category, I would like an LLM to determine if there are any duplicate groups, which I will then remove. Then, I would like to go through each name and have an LLM determine what groups it belongs to.

To account for user input of non-existing names or groups, I would like to generate 5 names and 5 groups that don't exist.

Then, since user input will vary from person to person, I would like to generate 5 variations of each name or group keyword (including non-existing that I generated) using an LLM.

When all variations are generated, I will then convert each keyword (including all names, groups, and variations thereof) into an embedding. Then, for each variant group (the name or group and its variations), I will calculate the cosine distance between each variant embedding (the base) and all embeddings outside of the variant group (options).

I will use these distances to determine a proper threshold strategy for determining semantic relevance. When considering a base and the desired options retrieved, for each base the strategy should ideally capture:
- existing group >>  all names belonging to the group 
- existing name >> just that name
- non-existing group or name >> nothing

To analyze the distances, I will perform the operations represented by this pseudocode:
for each name in names
    for each variation in name['variations]
        order 
        inc. name, order distances_to_variation
        find the index and distance of the name
        from the distances up to and including the name index, calculate the percentage that share the same group
5. for each group in groups
    for each variation in group['variations]
        inc. name, order distances_to_variation
        find the index and distance required to capture all members of the group
        from the distances up to and including the largest index fo a group member, calculate the percentage that share the same group
        calculate num_of_nonmembers_before_largest_index / largest_index_of_group_member

Once I jabe 
- plot the index, distance, and membership percentage for each name to see any trends
- do the same for groups
- algorithmically determine the optimal threshold by testing different thresholds and seeing 
    - the success rate in retrieving a single name (for each name, success is binary)
    - the success rate in retrieving members of a group (for each group, the success rate can range from 0 to 1)


Since there are many steps here, please create code that keeps track of the information flow in an intelligble and organized way.


So, we have in terms of steps:
1. input label(s) --> input embedding(s)
2. compare input embedding(s) to existing embeddings --> distances
3. reject any distances above threshold
4. if len(remaining embeddings) == 0 --> "nothing found"
   if len(remaining embeddings) == 1 --> retrieve group/name
   if len(remaining embeddings) >= 1 --> retrive name and groups
Depending on what I'm seeing, I might be able to partition the thresholds such that there is a middle partition whereupon the user is asked to elucidate their input.



Referring to a group or name as a label
1. get the embeddings for all labels
2. for each name in names
    for each variation in name['variations]
        dist = dist_func(variation, labels_besides_name)
3. do the same for groups
4. for each name in names
    for each variation in name['variations]
        inc. name, order distances_to_variation
        find the index and distance of the name
        from the distances up to and including the name index, calculate the percentage that share the same group
5. for each group in groups
    for each variation in group['variations]
        inc. name, order distances_to_variation
        find the index and distance required to capture all members of the group
        from the distances up to and including the largest index fo a group member, calculate the percentage that share the same group
        calculate num_of_nonmembers_before_largest_index / largest_index_of_group_member





In addition to the 5 variations

In [19]:
names.keys()

dict_keys(['scheduled', 'unscheduled', 'meals', 'values', 'sleep', 'sensations', 'scores', 'ingredients'])

In [ ]:

MODEL = 'gpt-4-1106-preview'
# MODEL = "gpt-3.5-turbo-1106" # this is faster!

client = OpenAI()

"""
# first I need to get all strings eligible for comparison
# this includes all event names, which will be given with event descriptions

for people, I will give the name of the person with their description
- I will include the name/type of people in the description

for locations, I will not offer that functionality right now


"""

def make_test(profile='llm_v1')
# transfer from the database
    collection = connect_to_collection()
    "profile_id": profile
    document = next(collection.find({})) 

# Embeddings

In [39]:
DEFAULT_EMBEDDING_MODEL = "text-embedding-ada-002"
def request_embedding(text, model=DEFAULT_EMBEDDING_MODEL):
    """
    Input can be a single string or a list of strings
    """
    client = OpenAI()

    if isinstance(text, str):
            text = [text]

    response = client.embeddings.create(input = text, model=model)

    embeddings = [embedding.embedding for embedding in response.data]

    if len(embeddings) == 1:
        return embeddings[0]
    else:
        return embeddings

        
emb = request_embedding(['hello', 'world'])


In [36]:

from openai import OpenAI
client = OpenAI()
text = "test\n"
e = client.embeddings.create(input = ["test"], model="text-embedding-ada-002")

In [37]:
e.data

[Embedding(embedding=[-0.0016985071124508977, -0.013903569430112839, 0.0016152469906955957, -0.01836981810629368, -0.007179654203355312, 0.018636250868439674, -0.009612604044377804, -0.02916734665632248, -0.006222601048648357, -0.014639764092862606, 0.011730040423572063, 0.007165631745010614, 0.0013803656911477447, -0.011856245808303356, -0.00708850659430027, 0.012992089614272118, 0.046948209404945374, 0.0009070974192582071, 0.017079723998904228, -0.028676548972725868, 0.010615231469273567, 0.007214711047708988, -0.0032112125772982836, 0.0014609965728595853, -0.030205029994249344, -0.014794014394283295, 0.0017396989278495312, -0.03704813867807388, -0.0031253232154995203, 0.002862396417185664, 0.01884659193456173, -0.028971027582883835, -0.026685316115617752, -0.029896529391407967, -0.023375945165753365, -0.008448714390397072, -0.003957924898713827, -0.014120921492576599, 0.017907068133354187, -0.0001308060745941475, 0.010573162697255611, 0.013223464600741863, 0.005416291765868664, -0.0

In [28]:
inpts = ["test{}".format(i) for i in range(10)] 

In [29]:
%%timeit
for i in inpts:
    e = client.embeddings.create(input = [i], model="text-embedding-ada-002")

1.43 s ± 264 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [30]:
%%timeit
client.embeddings.create(input = inpts, model="text-embedding-ada-002")

309 ms ± 162 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [21]:
dict(e).keys()

dict_keys(['data', 'model', 'object', 'usage'])